# TS-ResNet50 x224 Feature Autoencoder Dimension Sweep

This notebook evaluates how feature autoencoder hidden dimension affects ResNet50 teacher-student distillation at x224 resolution.

**Research Question:** Can larger feature autoencoder bottlenecks fix the resolution degradation observed at x224 for large backbones?

**Dimensions tested:** [64, 128, 256, 512, 768]

## Submission Context

- Dataset: `data/processed/x224/wm811k/metadata_50k_5pct.csv`
- Experiment config: `experiments/anomaly_detection/teacher_student/resnet50/x224/feature_autoencoder_dim_sweep/train_config.toml`
- Artifact root: `experiments/anomaly_detection/teacher_student/resnet50/x224/feature_autoencoder_dim_sweep/artifacts/ts_resnet50_x224_ae_dim_sweep/`
- Per-dimension outputs:
  - Training: `artifacts/ts_resnet50_x224_ae_dim_sweep/ae_dim_64/`, `ae_dim_128/`, etc.
  - Checkpoints: `checkpoints/best_model.pt` in each dimension folder
  - Evaluation: `results/evaluation/summary.json` in each dimension folder

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
from IPython.display import display

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
REPO_ROOT = None
for candidate in candidate_roots:
    if (candidate / "src" / "wafer_defect").exists() and (candidate / "configs").exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    raise RuntimeError("Could not locate repo root containing src/wafer_defect and configs/")

SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from wafer_defect.config import load_toml
from wafer_defect.data.wm811k import WaferMapDataset
from wafer_defect.evaluation.reconstruction_metrics import summarize_threshold_metrics, sweep_threshold_metrics
from wafer_defect.models.ts_distillation import build_ts_distillation_from_config
from wafer_defect.scoring import spatial_max, spatial_mean, topk_spatial_mean

REPO_ROOT

In [ ]:
# ===== CONFIGURATION =====
ARTIFACT_DIR = REPO_ROOT / 'experiments/anomaly_detection/teacher_student/resnet50/x224/feature_autoencoder_dim_sweep/artifacts/ts_resnet50_x224_ae_dim_sweep'
CONFIG_PATH = REPO_ROOT / 'experiments/anomaly_detection/teacher_student/resnet50/x224/feature_autoencoder_dim_sweep/train_config.toml'
THRESHOLD_QUANTILE = 0.95

# Dimension sweep configuration
AE_HIDDEN_DIMS = [64, 128, 256, 512, 768]

# ===== MAIN CONTROL FLAGS =====
RETRAIN = False
RUN_LOCAL_TRAINING = RETRAIN
RUN_DEFAULT_EVALUATION = RETRAIN

config = load_toml(CONFIG_PATH)
image_size = int(config['data'].get('image_size', 224))
batch_size = int(config['data'].get('batch_size', 256))
num_workers = int(config['data'].get('num_workers', 8))

requested_device = str(config['training'].get('device', 'auto'))
resolved_device_name = 'cuda' if requested_device == 'auto' and torch.cuda.is_available() else requested_device
device = torch.device(resolved_device_name)

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Artifact dir: {ARTIFACT_DIR}')
print(f'Config: {CONFIG_PATH}')
print(f'Device: {device}')
print(f'AE hidden dims to test: {AE_HIDDEN_DIMS}')

In [ ]:
def stream_command(command, cwd):
    print('Running:', ' '.join(str(part) for part in command))
    process = subprocess.Popen(
        [str(part) for part in command],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

def move_if_exists(source, destination):
    source = Path(source)
    destination = Path(destination)
    if source.exists():
        destination.parent.mkdir(parents=True, exist_ok=True)
        if destination.exists():
            destination.unlink()
        source.replace(destination)

def normalize_run_artifacts(run_dir):
    run_dir = Path(run_dir)
    checkpoint_dir = run_dir / 'checkpoints'
    results_dir = run_dir / 'results'
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    results_dir.mkdir(parents=True, exist_ok=True)

    for filename in ['best_model.pt', 'latest_checkpoint.pt', 'last_model.pt']:
        root_file = run_dir / filename
        target_file = checkpoint_dir / filename
        if root_file.exists() and root_file.resolve() != target_file.resolve():
            move_if_exists(root_file, target_file)

    for checkpoint_file in run_dir.glob('checkpoint_epoch_*.pt'):
        move_if_exists(checkpoint_file, checkpoint_dir / checkpoint_file.name)

    for filename in ['history.json', 'summary.json']:
        root_file = run_dir / filename
        target_file = results_dir / filename
        if root_file.exists() and root_file.resolve() != target_file.resolve():
            move_if_exists(root_file, target_file)

In [ ]:
# Iterate through each hidden dimension and train/evaluate
sweep_results = []

for ae_hidden_dim in AE_HIDDEN_DIMS:
    dim_name = f'ae_dim_{ae_hidden_dim}'
    dim_artifact_dir = ARTIFACT_DIR / dim_name
    dim_checkpoint_dir = dim_artifact_dir / 'checkpoints'
    dim_results_dir = dim_artifact_dir / 'results'
    dim_evaluation_dir = dim_results_dir / 'evaluation'
    dim_plots_dir = dim_artifact_dir / 'plots'
    dim_checkpoint_path = dim_checkpoint_dir / 'best_model.pt'
    dim_summary_path = dim_evaluation_dir / 'summary.json'

    for path in [dim_checkpoint_dir, dim_results_dir, dim_evaluation_dir, dim_plots_dir]:
        path.mkdir(parents=True, exist_ok=True)

    print(f'\n{"="*80}')
    print(f'Processing AE hidden dim = {ae_hidden_dim}')
    print(f'Artifact dir: {dim_artifact_dir}')
    print(f'{"="*80}\n')

    if RUN_LOCAL_TRAINING and not dim_checkpoint_path.exists():
        # Create per-dimension config
        dim_config = load_toml(CONFIG_PATH)
        dim_config['model']['feature_autoencoder_hidden_dim'] = ae_hidden_dim
        dim_config['run']['output_dir'] = str(dim_artifact_dir)

        # Write temp config
        dim_config_path = dim_artifact_dir / 'train_config_temp.toml'
        from wafer_defect.config import save_toml
        save_toml(dim_config, dim_config_path)

        stream_command(
            [
                sys.executable,
                '-u',
                REPO_ROOT / 'scripts' / 'train_ts_distillation.py',
                '--config',
                dim_config_path,
            ],
            cwd=REPO_ROOT,
        )
        normalize_run_artifacts(dim_artifact_dir)
    elif not dim_checkpoint_path.exists():
        print(f'Skipping training. Expecting checkpoint at {dim_checkpoint_path}')

    if not dim_checkpoint_path.exists():
        print(f'Warning Checkpoint not found for ae_dim={ae_hidden_dim}. Skipping evaluation.')
        continue

    # Load evaluation summary if available
    if dim_summary_path.exists():
        with open(dim_summary_path) as f:
            eval_summary = json.load(f)
        metrics = eval_summary.get('metrics_at_validation_threshold', {})
        print(f' Loaded cached evaluation for ae_dim={ae_hidden_dim}')
        print(f"  F1={metrics.get('f1', 'N/A'):.4f}, AUROC={metrics.get('auroc', 'N/A'):.4f}, AUPRC={metrics.get('auprc', 'N/A'):.4f}")

        sweep_results.append({
            'ae_hidden_dim': ae_hidden_dim,
            'f1': metrics.get('f1', np.nan),
            'auroc': metrics.get('auroc', np.nan),
            'auprc': metrics.get('auprc', np.nan),
            'precision': metrics.get('precision', np.nan),
            'recall': metrics.get('recall', np.nan),
            'threshold': eval_summary.get('threshold', np.nan),
        })
    else:
        print(f'Warning Evaluation summary not found for ae_dim={ae_hidden_dim}. Run evaluation or')

# Display sweep results
if sweep_results:
    sweep_df = pd.DataFrame(sweep_results)
    print(f'\n{"="*80}')
    print('Dimension Sweep Summary')
    print(f'{"="*80}\n')
    display(sweep_df)

    # Find best dimension by F1
    best_idx = sweep_df['f1'].idxmax()
    best_dim = sweep_df.loc[best_idx, 'ae_hidden_dim']
    best_f1 = sweep_df.loc[best_idx, 'f1']
    print(f'\n Best AE hidden dim: {int(best_dim)} (F1={best_f1:.4f})')

    # Save sweep summary
    sweep_df.to_csv(ARTIFACT_DIR / 'ae_dimension_sweep_summary.csv', index=False)
    print(f' Sweep summary saved to ae_dimension_sweep_summary.csv')
else:
    print('No results to display. Run training or load checkpoints.')

In [ ]:
# Plot dimension sweep results
if sweep_results:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    sweep_df_sorted = sweep_df.sort_values('ae_hidden_dim')

    axes[0].plot(sweep_df_sorted['ae_hidden_dim'], sweep_df_sorted['f1'], marker='o', linewidth=2, markersize=8, color='#457b9d')
    axes[0].set_xlabel('AE Hidden Dimension')
    axes[0].set_ylabel('F1')
    axes[0].set_title('F1 vs AE Hidden Dimension')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(sweep_df_sorted['ae_hidden_dim'], sweep_df_sorted['auroc'], marker='s', linewidth=2, markersize=8, color='#2a9d8f')
    axes[1].set_xlabel('AE Hidden Dimension')
    axes[1].set_ylabel('AUROC')
    axes[1].set_title('AUROC vs AE Hidden Dimension')
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(sweep_df_sorted['ae_hidden_dim'], sweep_df_sorted['auprc'], marker='^', linewidth=2, markersize=8, color='#e76f51')
    axes[2].set_xlabel('AE Hidden Dimension')
    axes[2].set_ylabel('AUPRC')
    axes[2].set_title('AUPRC vs AE Hidden Dimension')
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(ARTIFACT_DIR / 'ae_dimension_sweep.png', dpi=200, bbox_inches='tight')
    plt.show()

    print(' Sweep plots saved to ae_dimension_sweep.png')

In [ ]:
# Comparison: x64 baseline vs best x224 dimension
print('\nComparison: ResNet50 x64 vs Best ResNet50 x224 (by AE dimension)')
print('='*70)

x64_baseline = {
    'F1': 0.488,
    'AUROC': 0.913,
    'AUPRC': 0.581,
    'AE_dim': 128,
}

if sweep_results:
    best_idx = sweep_df['f1'].idxmax()
    best_result = sweep_df.iloc[best_idx]

    comparison = pd.DataFrame([
        {
            'Configuration': f"ResNet50 x64 (AE_dim={x64_baseline['AE_dim']})",
            'F1': x64_baseline['F1'],
            'AUROC': x64_baseline['AUROC'],
            'AUPRC': x64_baseline['AUPRC'],
        },
        {
            'Configuration': f"ResNet50 x224 (AE_dim={int(best_result['ae_hidden_dim'])})",
            'F1': best_result['f1'],
            'AUROC': best_result['auroc'],
            'AUPRC': best_result['auprc'],
        },
    ])

    display(comparison)

    # Calculate improvements
    f1_delta = best_result['f1'] - x64_baseline['F1']
    f1_pct = (f1_delta / x64_baseline['F1']) * 100

    print(f'\nResolution Impact (x224 vs x64):')
    print(f'  F1: {f1_delta:+.4f} ({f1_pct:+.1f}%)')
    print(f'  AUROC: {best_result["auroc"] - x64_baseline["AUROC"]:+.4f}')
    print(f'  AUPRC: {best_result["auprc"] - x64_baseline["AUPRC"]:+.4f}')